# ContourPose ADD(-S) LaTeX Table Generator

Reads per-object evaluation CSVs from `results/rtless_paper/` and produces a LaTeX comparison table
matching the format in the paper submission.

**Note on object mapping (README §3):**  
The README maps paper obj7 *and* obj8 both to code `obj18`, which is a typo.  
Code `obj6` exists in `results/` but is absent from the README mapping — it is almost certainly paper obj8.  
Verify and adjust `PAPER_TO_CODE` below if needed.

In [1]:
import pandas as pd
import json
from pathlib import Path

In [2]:
# ADD(-S) % values reported in the ContourPose paper (Table 1)
REPORTED = {
    'Obj 1':  100.00,
    'Obj 2':   97.54,
    'Obj 3':   95.35,
    'Obj 4':   88.14,
    'Obj 5':   90.70,
    'Obj 6':   96.71,
    'Obj 7':   91.82,
    'Obj 8':   95.31,
    'Obj 9':   93.50,
    'Obj 10':  92.30,
}

# Paper label → code folder name under results/rtless_paper/
# README lists paper obj7→obj18 AND obj8→obj18 (duplicate).
# Code obj6 has results but is missing from the README — assumed to be paper obj8.
PAPER_TO_CODE = {
    'Obj 1':  'obj1',
    'Obj 2':  'obj2',
    'Obj 3':  'obj3',
    'Obj 4':  'obj7',
    'Obj 5':  'obj13',
    'Obj 6':  'obj16',
    'Obj 7':  'obj18',
    'Obj 8':  'obj6',   # README typo: listed as obj18; obj6 is the only unmapped code object
    'Obj 9':  'obj21',
    'Obj 10': 'obj32',
}

In [3]:
RESULTS_DIR = Path("../results/rtless_paper")

rows = []
for paper_label, code_label in PAPER_TO_CODE.items():
    csv_path = RESULTS_DIR / code_label / "masked_detailed.csv"
    meta_path = RESULTS_DIR / code_label / "metadata.json"

    if not csv_path.exists():
        print(f"WARNING: {csv_path} not found — skipping {paper_label}")
        continue

    df = pd.read_csv(csv_path)
    add_rate = df["add_pass"].mean() * 100
    n_instances = len(df)

    meta = {}
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())

    rows.append({
        "Paper label":   paper_label,
        "Code object":   code_label,
        "N instances":   n_instances,
        "Reported (%)": REPORTED[paper_label],
        "Reproduced (%)": round(add_rate, 2),
        "PECP used":     meta.get("use_pecp"),
        "Epoch":         meta.get("epoch"),
    })

summary = pd.DataFrame(rows)
summary

,Paper label,Code object,N instances,Reported (%),Reproduced (%),PECP used,Epoch
0,Obj 1,obj1,1248,100.00,100.00,True,150
1,Obj 2,obj2,1248,97.54,99.20,True,140
2,Obj 3,obj3,1248,95.35,99.12,True,150
3,Obj 4,obj7,1248,88.14,99.76,True,150
4,Obj 5,obj13,1248,90.70,99.92,True,150
5,Obj 6,obj16,832,96.71,100.00,True,150
6,Obj 7,obj18,832,91.82,100.00,True,170
7,Obj 8,obj6,1248,95.31,95.03,True,150
8,Obj 9,obj21,832,93.50,97.60,True,140
9,Obj 10,obj32,416,92.30,98.32,True,160


In [4]:
def _fmt(v: float, bold: bool) -> str:
    s = f"{v:.2f}"
    return rf"\textbf{{{s}}}" if bold else s


def make_latex_table(df: pd.DataFrame) -> str:
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\caption{Comparison on Reflective Metal Parts Dataset Using the ADD(-S) Metric}",
        r"\label{tab:contourpose_comparison}",
        r"\begin{tabular}{lcc}",
        r"\toprule",
        r"\textbf{Methods} & \textbf{ContourPose (Reported)} & \textbf{Reproduced (Author's Data)} \\\\",
        r"\midrule",
    ]

    for _, row in df.iterrows():
        rep   = row["Reported (%)"]
        repro = row["Reproduced (%)"]
        lines.append(
            f"{row['Paper label']:6s} & "
            f"{_fmt(rep,   rep   >= repro):25s} & "
            f"{_fmt(repro, repro >  rep  )} \\\\"
        )

    lines.append(r"\midrule")

    # Average (All)
    all_rep   = df["Reported (%)"].mean()
    all_repro = df["Reproduced (%)"].mean()
    lines.append(
        f"Average (All)       & "
        f"{_fmt(all_rep,   all_rep   >= all_repro):25s} & "
        f"{_fmt(all_repro, all_repro >  all_rep  )} \\\\"
    )

    # Average (Obj 2--10)
    tail = df.iloc[1:]
    t_rep   = tail["Reported (%)"].mean()
    t_repro = tail["Reproduced (%)"].mean()
    lines.append(
        f"Average (Obj 2--10) & "
        f"{_fmt(t_rep,   t_rep   >= t_repro):25s} & "
        f"{_fmt(t_repro, t_repro >  t_rep  )} \\\\"
    )

    lines += [
        r"\bottomrule",
        r"\addlinespace[4pt]",
        r"\multicolumn{3}{p{0.85\linewidth}}{\footnotesize \textbf{Bold} = best per metric. "
        r"Per-object values differ from reported results, but overall averages are comparable, "
        r"validating our implementation.} \\\\",
        r"\end{tabular}",
        r"\end{table}",
    ]

    return "\n".join(lines)


latex = make_latex_table(summary)
print(latex)

\begin{table}[ht]
\centering
\caption{Comparison on Reflective Metal Parts Dataset Using the ADD(-S) Metric}
\label{tab:contourpose_comparison}
\begin{tabular}{lcc}
\toprule
\textbf{Methods} & \textbf{ContourPose (Reported)} & \textbf{Reproduced (Author's Data)} \\\\
\midrule
Obj 1  & \textbf{100.00}           & 100.00 \\
Obj 2  & 97.54                     & \textbf{99.20} \\
Obj 3  & 95.35                     & \textbf{99.12} \\
Obj 4  & 88.14                     & \textbf{99.76} \\
Obj 5  & 90.70                     & \textbf{99.92} \\
Obj 6  & 96.71                     & \textbf{100.00} \\
Obj 7  & 91.82                     & \textbf{100.00} \\
Obj 8  & \textbf{95.31}            & 95.03 \\
Obj 9  & 93.50                     & \textbf{97.60} \\
Obj 10 & 92.30                     & \textbf{98.32} \\
\midrule
Average (All)       & 94.14                     & \textbf{98.90} \\
Average (Obj 2--10) & 93.49                     & \textbf{98.77} \\
\bottomrule
\addlinespace[4pt]
\multicolumn

In [5]:
# Optionally write to a .tex file
out_path = Path("../results/table_add.tex")
out_path.write_text(latex)
print(f"Saved to {out_path.resolve()}")

Saved to /contourpose-baseline/results/table_add.tex
